# 02 — Fine-tune SegFormer on FoodSeg103 (GPU job 1)

Trains the on-device food segmenter. Every public FoodSeg103 checkpoint measures at mIoU ≤ 0.05 (docs/MODELS.md), so we train our own.

- **Runtime:** H100 — B0 ~2–3 h, B1 ~4–6 h (60 epochs, batch 32). Select the GPU runtime before running.
- **Targets:** mIoU ≥ 0.25 (mit-b0) / ≥ 0.32 (mit-b1)
- **Everything persists to Drive** — the FoodSeg103 dataset cache, the pretrained backbone, and per-epoch checkpoints — so a disconnect loses one epoch at most; rerun and `Trainer` resumes from the last checkpoint. Notebook 01 is NOT required for this one.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/physics-powered-portion-estimation'

# HF caches stay on LOCAL disk. The datasets library memory-maps its Arrow
# files, and mmap over Google Drive's FUSE mount fails — so the dataset and
# the (tiny) pretrained backbone live locally and re-download in seconds on a
# fresh VM. Only the trained checkpoints (OUT, below) persist to Drive; that's
# the output worth keeping.
os.environ['HF_HOME'] = '/content/hf-cache'
os.environ['HF_DATASETS_CACHE'] = '/content/hf-datasets'

MODEL = 'nvidia/mit-b0'   # or 'nvidia/mit-b1'
EPOCHS = 60
BATCH = 32
OUT = f"{DRIVE_ROOT}/checkpoints/segformer-{MODEL.split('/')[-1]}-food"

!nvidia-smi | head -12

In [ ]:
!git clone --depth 1 https://github.com/Hakeem-Hannoon/physics-powered-portion-estimation.git /content/ppe 2>/dev/null || (cd /content/ppe && git pull)
%pip -q install transformers datasets evaluate accelerate timm

In [ ]:
%cd /content/ppe
!python model/train/segformer_foodseg103.py \
  --model "{MODEL}" --epochs {EPOCHS} --batch-size {BATCH} \
  --output "{OUT}"

In [ ]:
# Result row for the README 'Testing set & results' table. Robust: reads the
# metric wherever it landed, never IndexErrors on missing checkpoint dirs.
import glob, json, os

def find_miou(out):
    p = os.path.join(out, 'eval_results.json')  # written by the training script
    if os.path.exists(p):
        m = json.load(open(p)).get('eval_mean_iou')
        if m is not None:
            return m
    states = [os.path.join(out, 'trainer_state.json')] + \
        sorted(glob.glob(f'{out}/checkpoint-*/trainer_state.json'))
    for sp in states:
        if not os.path.exists(sp):
            continue
        d = json.load(open(sp))
        if d.get('best_metric') is not None:
            return d['best_metric']
        ious = [e['eval_mean_iou'] for e in d.get('log_history', []) if 'eval_mean_iou' in e]
        if ious:
            return max(ious)
    return None

best = find_miou(OUT)
if best is None:
    print('No metric file found. Contents of OUT:')
    print(sorted(os.listdir(OUT)) if os.path.isdir(OUT) else 'OUT missing')
    print('If OUT holds model.safetensors the model trained fine — '
          'scroll the training cell for the last eval_mean_iou.')
else:
    print(f'best mIoU: {best:.4f}')
    print(f'| FoodSeg103 val | mIoU | public <= 0.05; B0 ~ 0.25, B1 ~ 0.32 | '
          f'**{best:.3f}** ({MODEL}) |')